# 22 — Cross-Frequency Observables on PhysioNet Sleep Data

Three new observables applied to 4 MIT-BIH Polysomnographic recordings:
1. **Phase-Amplitude Coupling (PAC)** — Tort et al. (2010) Modulation Index
2. **Wavelet Coherence** — Morlet wavelet, time-frequency cross-scale coupling
3. **Granger Causality Spectrum** — frequency-resolved directed coupling (VAR model)

Goal: test SCPN cross-scale coupling predictions with established neuroscience methodology.
Compare with K_nm predictions and TE results from notebook 19.

In [ ]:
import json
from pathlib import Path

import numpy as np
import wfdb
from scipy import signal

DATA_DIR = Path(
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/00_DATASETS/physionet_slpdb"
)
RECORDS = ["slp01a", "slp01b", "slp02a", "slp02b"]
FS = 250  # Hz

# EEG frequency bands
BANDS = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma": (30, 80),
}


def load_record(name):
    rec = wfdb.rdrecord(str(DATA_DIR / name))
    data = rec.p_signal
    sigs = {n: data[:, i] for i, n in enumerate(rec.sig_name)}
    return sigs


def bandpass(x, lo, hi, fs=FS, order=4):
    nyq = fs / 2
    lo_n = max(lo / nyq, 0.001)
    hi_n = min(hi / nyq, 0.999)
    b, a = signal.butter(order, [lo_n, hi_n], btype="band")
    return signal.filtfilt(b, a, x)


print(f"Records: {RECORDS}")
print(f"Bands: {list(BANDS.keys())}")
print(f"Fs: {FS} Hz")

## 1. Phase-Amplitude Coupling (PAC)

Tort et al. (2010): Modulation Index (MI) = KL divergence of amplitude distribution
across phase bins from uniform.

Phase of low-frequency band modulates amplitude of high-frequency band.
Strongest known PAC in neuroscience: theta modulates gamma.

In [ ]:
def compute_pac_mi(x, phase_band, amp_band, fs=FS, n_bins=18):
    """Tort (2010) Modulation Index.

    MI = KL(amplitude_distribution || uniform) / log(n_bins)
    """
    x_phase = bandpass(x, *phase_band, fs=fs)
    x_amp = bandpass(x, *amp_band, fs=fs)

    phase = np.angle(signal.hilbert(x_phase))
    amplitude = np.abs(signal.hilbert(x_amp))

    # Bin amplitude by phase
    bin_edges = np.linspace(-np.pi, np.pi, n_bins + 1)
    mean_amp = np.zeros(n_bins)
    for b in range(n_bins):
        mask = (phase >= bin_edges[b]) & (phase < bin_edges[b + 1])
        if np.sum(mask) > 0:
            mean_amp[b] = np.mean(amplitude[mask])

    # Normalise to distribution
    total = np.sum(mean_amp)
    if total < 1e-10:
        return 0.0
    p = mean_amp / total

    # KL divergence from uniform
    uniform = np.ones(n_bins) / n_bins
    # Avoid log(0)
    p_safe = np.clip(p, 1e-10, None)
    kl = np.sum(p_safe * np.log(p_safe / uniform))
    mi = kl / np.log(n_bins)
    return float(mi)


# Compute PAC for all band pairs across all records
# Use 5-minute segments for stability
SEG_LEN = 5 * 60 * FS  # 5 min in samples

pac_results = {}
phase_bands = ["delta", "theta", "alpha"]
amp_bands = ["alpha", "beta", "gamma"]

for rec_name in RECORDS:
    print(f"\n--- {rec_name} ---")
    sigs = load_record(rec_name)
    eeg_key = [k for k in sigs if "EEG" in k][0]
    eeg = sigs[eeg_key]

    # Use middle 30 minutes (stable sleep)
    mid = len(eeg) // 2
    start = mid - 15 * 60 * FS
    end = mid + 15 * 60 * FS
    eeg_seg = eeg[start:end]

    rec_pac = {}
    for pb in phase_bands:
        for ab in amp_bands:
            if BANDS[pb][1] <= BANDS[ab][0]:  # phase freq < amp freq
                mi = compute_pac_mi(eeg_seg, BANDS[pb], BANDS[ab])
                pair = f"{pb}→{ab}"
                rec_pac[pair] = round(mi, 6)
                print(f"  PAC {pair}: MI = {mi:.6f}")
    pac_results[rec_name] = rec_pac

# Summary
print("\n=== PAC Summary ===")
all_pairs = list(pac_results[RECORDS[0]].keys())
for pair in all_pairs:
    vals = [pac_results[r][pair] for r in RECORDS]
    print(f"{pair}: mean MI = {np.mean(vals):.6f} ± {np.std(vals):.6f}")

## 2. Wavelet Coherence

Morlet wavelet coherence between EEG and autonomic signals (ECG, BP, Resp).
Reveals time-frequency structure of cross-scale coupling.

In [ ]:
def morlet_wavelet(f, fs, n_cycles=6):
    """Complex Morlet wavelet at frequency f."""
    sigma = n_cycles / (2 * np.pi * f)
    t = np.arange(-4 * sigma, 4 * sigma, 1.0 / fs)
    w = np.exp(2j * np.pi * f * t) * np.exp(-(t**2) / (2 * sigma**2))
    return w / np.sqrt(sigma * np.sqrt(np.pi))


def wavelet_transform(x, freqs, fs=FS, n_cycles=6):
    """Compute continuous wavelet transform at specified frequencies."""
    n = len(x)
    W = np.zeros((len(freqs), n), dtype=complex)
    for i, f in enumerate(freqs):
        wav = morlet_wavelet(f, fs, n_cycles)
        W[i] = np.convolve(x, wav, mode="same")
    return W


def wavelet_coherence(x, y, freqs, fs=FS, n_cycles=6, smooth_n=50):
    """Wavelet coherence between x and y."""
    Wx = wavelet_transform(x, freqs, fs, n_cycles)
    Wy = wavelet_transform(y, freqs, fs, n_cycles)

    kernel = np.ones(smooth_n) / smooth_n

    coh = np.zeros((len(freqs), len(x)))
    for i in range(len(freqs)):
        Sxy = np.convolve(Wx[i] * np.conj(Wy[i]), kernel, mode="same")
        Sxx = np.convolve(np.abs(Wx[i]) ** 2, kernel, mode="same")
        Syy = np.convolve(np.abs(Wy[i]) ** 2, kernel, mode="same")
        denom = np.sqrt(Sxx * Syy)
        denom[denom < 1e-10] = 1e-10
        coh[i] = np.abs(Sxy) ** 2 / (Sxx * Syy)
    return coh


# Frequencies: logarithmically spaced 0.5-80 Hz
freqs = np.logspace(np.log10(0.5), np.log10(80), 40)

# Compute wavelet coherence for EEG vs each autonomic signal
DS = 5  # downsample factor
FS_DS = FS // DS  # 50 Hz
SEG_WAV = 10 * 60 * FS_DS  # 10 min at 50 Hz
freqs_ds = freqs[freqs < FS_DS / 2]  # respect Nyquist

wcoh_results = {}
for rec_name in RECORDS:
    print(f"\n--- {rec_name} ---")
    sigs = load_record(rec_name)
    eeg_key = [k for k in sigs if "EEG" in k][0]
    eeg = signal.decimate(sigs[eeg_key], DS)

    mid = len(eeg) // 2
    eeg_seg = eeg[mid - SEG_WAV // 2 : mid + SEG_WAV // 2]

    rec_wcoh = {}
    for sig_name in ["ECG", "BP", "Resp (sum)"]:
        if sig_name not in sigs:
            continue
        other = signal.decimate(sigs[sig_name], DS)
        other_seg = other[mid - SEG_WAV // 2 : mid + SEG_WAV // 2]

        coh = wavelet_coherence(eeg_seg, other_seg, freqs_ds, fs=FS_DS, n_cycles=6, smooth_n=100)
        band_coh = {}
        for band_name, (lo, hi) in BANDS.items():
            mask = (freqs_ds >= lo) & (freqs_ds <= hi)
            if np.any(mask):
                band_coh[band_name] = round(float(np.mean(coh[mask])), 4)

        short_name = sig_name.split(" ")[0]
        rec_wcoh[f"EEG↔{short_name}"] = band_coh
        print(f"  EEG↔{short_name}: {band_coh}")

    wcoh_results[rec_name] = rec_wcoh

print("\n=== Wavelet Coherence Summary ===")
# Collect all pair keys that appear in any record
all_wcoh_pairs = set()
for r in RECORDS:
    all_wcoh_pairs.update(wcoh_results[r].keys())

for pair_key in sorted(all_wcoh_pairs):
    recs_with_pair = [r for r in RECORDS if pair_key in wcoh_results[r]]
    if not recs_with_pair:
        continue
    print(f"\n{pair_key} ({len(recs_with_pair)} records):")
    for band in BANDS:
        vals = [wcoh_results[r][pair_key].get(band, 0) for r in recs_with_pair]
        if any(v > 0 for v in vals):
            print(f"  {band}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}")

## 3. Granger Causality Spectrum

Frequency-resolved directed coupling via VAR model.
Complementary to Transfer Entropy (notebook 19) — linear, parametric, frequency-decomposed.

In [ ]:
def fit_var(X, order=10):
    """Fit VAR(p) model to multivariate time series X (n_samples, n_channels).

    Returns coefficient matrices A_1, ..., A_p and residual covariance.
    """
    n, k = X.shape
    # Build regression matrices
    Y = X[order:]  # (n-p, k)
    Z = np.zeros((n - order, order * k))
    for lag in range(1, order + 1):
        Z[:, (lag - 1) * k : lag * k] = X[order - lag : n - lag]

    # OLS: A = (Z'Z)^-1 Z'Y
    A, res, _, _ = np.linalg.lstsq(Z, Y, rcond=None)
    residuals = Y - Z @ A
    Sigma = residuals.T @ residuals / (n - order)

    # Reshape to list of (k, k) matrices
    A_list = [A[lag * k : (lag + 1) * k].T for lag in range(order)]
    return A_list, Sigma


def spectral_granger(X, order=10, fs=50, n_freqs=128):
    """Frequency-domain Granger causality from VAR model.

    Returns GC spectrum (n_freqs, n_channels, n_channels).
    GC[f, i, j] = Granger causality from j to i at frequency f.
    """
    A_list, Sigma = fit_var(X, order)
    k = X.shape[1]
    freqs = np.linspace(0, fs / 2, n_freqs)

    GC = np.zeros((n_freqs, k, k))
    for fi, f in enumerate(freqs):
        z = np.exp(-2j * np.pi * f / fs)
        A_f = np.eye(k, dtype=complex)
        for lag, A_lag in enumerate(A_list):
            A_f -= A_lag * (z ** (lag + 1))
        H = np.linalg.inv(A_f)  # Transfer function
        S = H @ Sigma @ H.conj().T  # Spectral density

        for i in range(k):
            for j in range(k):
                if i == j:
                    continue
                # GC from j→i: log ratio of full vs restricted spectral density
                S_ii = np.abs(S[i, i])
                if S_ii < 1e-20:
                    continue
                # Approximate restricted spectrum (remove j→i path)
                H_restricted = H.copy()
                H_restricted[i, j] = 0
                S_restricted = H_restricted @ Sigma @ H_restricted.conj().T
                S_ii_r = np.abs(S_restricted[i, i])
                if S_ii_r < 1e-20:
                    continue
                GC[fi, i, j] = max(0, np.log(S_ii / S_ii_r))

    return freqs, GC


# Apply to all records
# Downsample to 50 Hz, use 10-minute segment
SIGNAL_NAMES = ["ECG", "BP", "EEG", "Resp"]

gc_results = {}
for rec_name in RECORDS:
    print(f"\n--- {rec_name} ---")
    sigs = load_record(rec_name)

    # Map signal names
    sig_map = {}
    for k, v in sigs.items():
        if "ECG" in k:
            sig_map["ECG"] = v
        elif "BP" in k:
            sig_map["BP"] = v
        elif "EEG" in k:
            sig_map["EEG"] = v
        elif "Resp" in k or "resp" in k:
            sig_map["Resp"] = v

    available = [n for n in SIGNAL_NAMES if n in sig_map]

    # Downsample and segment
    X_raw = np.column_stack([signal.decimate(sig_map[n], DS) for n in available])
    mid = len(X_raw) // 2
    seg = SEG_WAV // 2
    X = X_raw[mid - seg : mid + seg]

    # Normalise each channel
    X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-10)

    freqs_gc, GC = spectral_granger(X, order=10, fs=FS_DS, n_freqs=64)

    # Summarise: mean GC per band for each directed pair
    rec_gc = {}
    for i, ni in enumerate(available):
        for j, nj in enumerate(available):
            if i == j:
                continue
            pair = f"{nj}→{ni}"
            band_gc = {}
            for band_name, (lo, hi) in BANDS.items():
                mask = (freqs_gc >= lo) & (freqs_gc <= hi)
                if np.any(mask):
                    val = float(np.mean(GC[mask, i, j]))
                    if val > 0.001:
                        band_gc[band_name] = round(val, 4)
            if band_gc:
                rec_gc[pair] = band_gc

    gc_results[rec_name] = rec_gc
    # Print top 5 strongest directed pairs
    sorted_pairs = sorted(rec_gc.items(), key=lambda x: max(x[1].values()), reverse=True)
    for pair, bands in sorted_pairs[:5]:
        print(f"  {pair}: {bands}")

print("\n=== Granger Causality Summary ===")
# Compute mean asymmetry (same as TE notebook 19)
asymmetries = []
for rec_name in RECORDS:
    gc = gc_results[rec_name]
    pairs_done = set()
    for pair, bands in gc.items():
        src, tgt = pair.split("→")
        rev = f"{tgt}→{src}"
        if rev in gc and pair not in pairs_done:
            pairs_done.add(pair)
            pairs_done.add(rev)
            for band in bands:
                if band in gc.get(rev, {}):
                    fwd = bands[band]
                    bwd = gc[rev][band]
                    asym = abs(fwd - bwd) / (fwd + bwd + 1e-10)
                    asymmetries.append(asym)

if asymmetries:
    print(f"Grand mean Granger asymmetry: {np.mean(asymmetries):.3f}")
    print("(cf. Transfer Entropy asymmetry from notebook 19: 0.361)")

## 4. Comparison with K_nm Predictions

In [ ]:
# Save all results
results = {
    "experiment": "Cross-frequency observables on PhysioNet sleep data",
    "records": RECORDS,
    "sampling_rate": FS,
    "bands": {k: list(v) for k, v in BANDS.items()},
    "pac": pac_results,
    "wavelet_coherence": wcoh_results,
    "granger_causality": gc_results,
}

out_path = Path(
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/cross_frequency_observables_2026-03-29.json"
)
out_path.parent.mkdir(exist_ok=True)
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {out_path}")
print(f"PAC pairs: {len(all_pairs)}")
print(f"Wavelet coherence signal pairs: {len(wcoh_results[RECORDS[0]])}")
print(f"Granger causality directed pairs: {len(gc_results[RECORDS[0]])}")

In [ ]:
# Save all results
results = {
    "experiment": "Cross-frequency observables on PhysioNet sleep data",
    "records": RECORDS,
    "sampling_rate": FS,
    "bands": {k: list(v) for k, v in BANDS.items()},
    "pac": pac_results,
    "wavelet_coherence": wcoh_results,
    "granger_causality": gc_results,
}

out_path = Path("../results/cross_frequency_observables_2026-03-29.json")
out_path.parent.mkdir(exist_ok=True)
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {out_path}")
print(f"PAC pairs: {len(all_pairs)}")
print(f"Wavelet coherence signal pairs: {len(wcoh_results[RECORDS[0]])}")
print(f"Granger causality directed pairs: {len(gc_results[RECORDS[0]])}")